In [ ]:
import math

# historical db
case_base = [
    {
        "case_name": "Typhoon Ursula (2019)",
        "problem": {"wind": 150, "rain": 200, "pop_density": 5000},
        "solution": {"level": 2, "shelter": 10000, "resources": "Standard"}
    },
    {
        "case_name": "Typhoon Odette (2021)",
        "problem": {"wind": 195, "rain": 300, "pop_density": 8000},
        "solution": {"level": 3, "shelter": 25000, "resources": "Max"}
    },
    {
        "case_name": "Typhoon Ondoy (2009)",
        "problem": {"wind": 130, "rain": 450, "pop_density": 15000},
        "solution": {"level": 3, "shelter": 40000, "resources": "Water-Rescue"}
    }
]

# normalization constants for distance scaling
MAXS = {"wind": 300.0, "rain": 500.0, "pop_density": 20000.0}

def calc_distance(p1, p2):
    # normalized euclidean distance
    return math.sqrt(sum(((p1[k] - p2[k]) / MAXS[k])**2 for k in MAXS))

def run_cbr(new_event):
    print(f"--- NEW EVENT: {new_event['case_name']} ---\nParams: {new_event['problem']}\n")
    prob = new_event["problem"]

    # 1. Retrieve: find closest match using min() and lambda
    best_match = min(case_base, key=lambda c: calc_distance(prob, c["problem"]))
    dist = calc_distance(prob, best_match["problem"])
    print(f"[RETRIEVE] Match: {best_match['case_name']} (Dist: {dist:.4f})")

    # 2. Reuse: extract base solution
    sol = best_match["solution"].copy()
    print(f"[REUSE] Base: {sol}")

    # 3. Revise: scale shelter capacity based on pop density ratio
    ratio = prob["pop_density"] / best_match["problem"]["pop_density"]
    sol["shelter"] = int(sol["shelter"] * ratio)
    print(f"[REVISE] Adapted: {sol}")

    # 4. Retain: save to db
    new_event["solution"] = sol
    case_base.append(new_event)
    print(f"[RETAIN] Saved. DB size: {len(case_base)}")

if __name__ == "__main__":
    new_case = {
        "case_name": "Typhoon Alpha (2026)",
        "problem": {"wind": 170, "rain": 250, "pop_density": 6500}
    }
    run_cbr(new_case)

--- NEW EVENT: Typhoon Alpha (2026) ---
Params: {'wind': 170, 'rain': 250, 'pop_density': 6500}

[RETRIEVE] Match: Typhoon Ursula (2019) (Dist: 0.1417)
[REUSE] Base: {'level': 2, 'shelter': 10000, 'resources': 'Standard'}
[REVISE] Adapted: {'level': 2, 'shelter': 13000, 'resources': 'Standard'}
[RETAIN] Saved. DB size: 4
